In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pandasql

  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=730cf04e75e0b961d4134ef2679ee52bd05ccf6197f77d5bd593210997647262
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandasql import sqldf
import sqlite3
from sklearn import linear_model
import statsmodels.api as sm
import math
from scipy import stats

In [ ]:
# Loading in S&P1500 stock factors dataframe
merged_df = pd.read_sas('/content/drive/merged_df.sas7bdat', encoding='latin-1')
merged_df

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/merged_df.sas7bdat'

In [ ]:
# Note that there are entries with existing permno but missing gvkeys and tickers.
len(merged_df.permno.unique())

In [ ]:
# To center monthid on Jan 2000 so it is less confusing
merged_df.monthid = merged_df.monthid - 241
merged_df[['yyyymm', 'monthid']]

In [ ]:
# To filter for stocks meeting price and market cap for Part 2 at the beginning of each year
filtered_merged_df = merged_df
filtered_merged_df['yearmonthid'] = merged_df.monthid - merged_df.monthid % 12

In [ ]:
year_groups = filtered_merged_df.groupby(['permno', 'yearmonthid'])

liquid = {'permno' : [], 'yearmonthid' : [], 'isLiquid' : []}

for name, group in year_groups :
  liquid['permno'].append(group.permno.iloc[0])
  liquid['yearmonthid'].append(group.yearmonthid.iloc[0])
  liquid['isLiquid'].append((group.PRC.iloc[0] > 5) & ((group.PRC.iloc[0] * group.SHROUT.iloc[0]) >= 100000))

# Created new column isLiquid which is True only if stock PRC > 5 and >= 100M market cap on Jan of each year.
isLiquid = pd.DataFrame(liquid)
filtered_merged_df = pd.merge(filtered_merged_df, isLiquid, on=['permno', 'yearmonthid'])
filtered_merged_df

In [ ]:
# Code to test whether isLiquid is correctly tracking stock price and cap requirements
filtered_merged_df.loc[filtered_merged_df.permno == 10026.0][['yyyymm', 'PRC', 'SHROUT', 'isLiquid']]

In [ ]:
# Step 4: Derive Factors
## Read in 1986-2019 data for chosen factors from WRDS (factor signals & financial ratio data tables used)
f1_df = pd.read_sas('momentum12.sas7bdat', encoding='latin-1')
f2_df = pd.read_sas('ITOA.sas7bdat', encoding='latin-1')
f3_df = pd.read_sas('DIV_P.sas7bdat', encoding='latin-1')
# f4_df = pd.read_sas('BtM.sas7bdat', encoding='latin-1')
# f5_df = pd.read_sas('REVSUP.sas7bdat', encoding='latin-1')
# f6_df = pd.read_sas('momentum.sas7bdat', encoding='latin-1')
f7_df = pd.read_sas('AccrualVol.sas7bdat', encoding='latin-1')
f8_df = pd.read_sas('beta.sas7bdat', encoding='latin-1')
# f9_df = pd.read_sas('ivol.sas7bdat', encoding='latin-1')
f10_df = pd.read_sas('FCF.sas7bdat', encoding='latin-1')
f11_df = pd.read_sas('CFVol.sas7bdat', encoding='latin-1')
# f12_df = pd.read_sas('trend_factor.sas7bdat', encoding='latin-1')
f13_df = pd.read_sas('SG.sas7bdat', encoding='latin-1')
f14_df = pd.read_sas('EV_EBITDA.sas7bdat', encoding='latin-1')
f15_df = pd.read_sas('AbsAccrual.sas7bdat', encoding='latin-1')
f16_df = pd.read_sas('CASHPROD.sas7bdat', encoding='latin-1')
# f17_df = pd.read_sas('REHOLDING.sas7bdat', encoding='latin-1')
f18_df = pd.read_sas('RD_P.sas7bdat', encoding='latin-1')
f19_df = pd.read_sas('momaccel.sas7bdat', encoding='latin-1')
# f20_df = pd.read_sas('FCF_P.sas7bdat', encoding='latin-1')
#f21_df = pd.read_sas('NOA.sas7bdat', encoding='latin-1')

ratio_factors = pd.read_csv('FinRatios.csv')
ratio_factors1 = pd.read_csv('FinRatios1.csv')
ratio_factors2 = pd.read_csv('FinRatios2.csv')
ratio_factors = pd.concat([ratio_factors, ratio_factors1, ratio_factors2], ignore_index=True)

In [ ]:
## Merge our factors from WRDS main signal database into our main df
WRDS_factors = [f1_df, f2_df, f3_df, f7_df, f8_df,
               f10_df, f11_df, f13_df, f14_df, f15_df, f16_df, f18_df, f19_df]
for i in WRDS_factors:
    i['monthid'] = (i['date'].dt.year - 2000) * 12 + i['date'].dt.month - 1
    filtered_merged_df = filtered_merged_df.merge(i, how='left', on=['permno', 'monthid'], suffixes=('main', 'factor')).drop(columns=['fdate', 'signal', 'date'])


In [ ]:
## Merge our factors from WRDS financial ratios into the main df
ratio_factors['monthid'] = (pd.to_datetime(ratio_factors['public_date']).dt.year - 2000) * 12 + pd.to_datetime(ratio_factors['public_date']).dt.month - 1
filtered_merged_df = filtered_merged_df.merge(ratio_factors, how='left', on =['permno', 'monthid'], suffixes=('main', 'finRatios'))

In [ ]:
## Use the 12-month momentum factor to and past period returns to derive the 12_1-month momentum factor
filtered_merged_df = filtered_merged_df[['permno', 'yyyymm', 'monthid', 'ticker', 'momentum12', 'ITOA', 'DIV_P', 'AccrualVol', 'beta', 'FCF', 'CFVol', 'SG', 'EV_EBITDA', 'AbsAccrual', 'CASHPROD', 'RD_P', 'momaccel', 'sue_NI', 'BM', 'EP', 'Cto', 'tail_2y', 'dp', 'aftret_invcapx', 'PRC', 'VOL', 'RET', 'SHROUT', 'ret_f1', 'ret_f2', 'ret_f3', 'ret_f4', 'ret_f5', 'ret_f6', 'ret_f7', 'ret_f8', 'ret_f9', 'ret_f10', 'ret_f11', 'ret_f12']]
filtered_merged_df = filtered_merged_df.sort_values(by=['permno', 'monthid']).reset_index(drop=True)

## We will need time t-1 returns
filtered_merged_df['ret_f-1'] = filtered_merged_df.groupby('permno')['RET'].shift(1)

twelve_one_moms = [np.nan]
for i in range(0, len(filtered_merged_df.permno)):
    if i < 1:
        continue
    elif pd.isna(filtered_merged_df.loc[i, 'momentum12']) or pd.isna(filtered_merged_df.loc[i, 'ret_f-1']):
        twelve_one_moms.append(np.nan)
    else:
        twelve_one_moms.append(((filtered_merged_df.loc[i, 'momentum12'] + 1) / (filtered_merged_df.loc[i, 'ret_f-1'] + 1)) - 1)
filtered_merged_df['momentum12_1'] = twelve_one_moms
filtered_merged_df = filtered_merged_df.drop(columns=['momentum12'])

In [ ]:
# Step 5: Data Filtering
filtered_merged_df_factors = filtered_merged_df.copy()
filtered_merged_df_factors = filtered_merged_df_factors.dropna()

# Winsorize factors
winsorize = ['momentum12_1', 'ITOA', 'DIV_P', 'AccrualVol', 'beta', 'FCF', 'CFVol', 'SG', 'EV_EBITDA', 'AbsAccrual', 'CASHPROD', 'RD_P', 'momaccel', 'sue_NI', 'BM', 'EP', 'Cto', 'tail_2y', 'dp', 'aftret_invcapx'] # all our factors
std_comparison = pd.DataFrame(columns=['Original', 'Winsorized'])

for var in winsorize:
    new_var = var + '_winsorized'
    filtered_merged_df_factors[new_var] = filtered_merged_df_factors.groupby('monthid')[var].transform(
        lambda x: x.clip(lower=x.quantile(0.01, interpolation="linear"), upper=x.quantile(0.99, interpolation="linear"))
    )
    std_comparison.loc[var] = [filtered_merged_df_factors[var].std(), filtered_merged_df_factors[new_var].std()]

print("Std before and after winsorization:")
print(round(std_comparison, 4))

# Create z-score standardized versions
winsorized_cols = [col for col in filtered_merged_df_factors.columns if '_winsorized' in col]


for var in winsorized_cols:
    new_var = var
    z_col = var + '_z'
    filtered_merged_df_factors[z_col] = filtered_merged_df_factors.groupby('monthid')[new_var].transform(
        lambda x: (x - x.mean()) / x.std()
    )
filtered_merged_df_factors

NameError: name 'filtered_merged_df' is not defined

In [ ]:
# Step 6 - Fama-MacBeth cross-sectional regressions

months = sorted(filtered_merged_df_factors['monthid'].unique())

# Pick the 75% point for seeing what the training cutoff should be in this case
cut_idx = int(len(months) * 0.75)  # e.g. 0.75 * 396 ≃ 297
TRAIN_CUTOFF = months[cut_idx]

print("Training sample covers monthid <=", TRAIN_CUTOFF,
      "(", cut_idx, "out of", len(months), "months )")

In [ ]:
# Define training sample
TRAIN_CUTOFF = 138.0
# copying over the table
train_df = filtered_merged_df_factors[filtered_merged_df_factors.monthid <= TRAIN_CUTOFF]

# Identify your z‑scored factor columns
factor_z_cols = [c for c in train_df.columns if c.endswith('_winsorized_z')]

# Run the monthly cross‑sectional regressions
monthly_coefs = []
for yyyymm, grp in train_df.groupby('yyyymm'):
    grp = grp.dropna(subset=['ret_f1'] + factor_z_cols)
    if len(grp) < len(factor_z_cols) + 1:
        continue
    Y = grp['ret_f1']
    X = sm.add_constant(grp[factor_z_cols])
    model = sm.OLS(Y, X, missing='drop').fit()
    # Convert the parameters Series to a dict, then inject the month, this gives month for df_fm_coefs
    row = model.params.to_dict()
    row['yyyymm'] = yyyymm
    monthly_coefs.append(row)

df_fm_coefs = pd.DataFrame(monthly_coefs) \
    .set_index('yyyymm') \
    .sort_index()

# Compute averages and t‑stats
T = len(df_fm_coefs)
means  = df_fm_coefs.mean()
stds   = df_fm_coefs.std()
se     = stds / np.sqrt(T)
tstats = means / se

fm_summary = pd.DataFrame({
    'avg_premia': means,
    'std_err':    se,
    't_stat':     tstats,
    'n_months':   T
}).round(4)

print("Fama–MacBeth Results on Training Sample (months ≤ 138):")
fm_summary

In [ ]:
# Gives the table with the coefs and winsorized z of all the facotrs for each entry
df_fm_coefs.head()

Re-Training withe the amount

In [ ]:
# step 8 - choosing factors

t_threshold = 1.5    # <<< raise/lower this to be more/less strict

# Identify factors meeting the threshold
# Drop the intercept—only look at real factors
raw_factors = fm_summary.index.drop('const')
mask = fm_summary.loc[raw_factors, 't_stat'].abs() >= t_threshold
factors_to_keep = raw_factors[mask].tolist()

print("✔︎ Factors with |t| ≥", t_threshold, ":")
for f in factors_to_keep:
    val = fm_summary.loc[f, ['avg_premia','t_stat']]
    print(f"   • {f:30s}  avg_premia={val.avg_premia:.4f}, t_stat={val.t_stat:.2f}")

In [ ]:
# step 9 - ReTrain with wanted factors
TRAIN_CUTOFF = 138.0
train_df = filtered_merged_df_factors[filtered_merged_df_factors.monthid <= TRAIN_CUTOFF].copy()

# Remove 4 manually decided factors
factors_to_keep.remove("DIV_P_winsorized_z")
factors_to_keep.remove("AbsAccrual_winsorized_z")
factors_to_keep.remove("EV_EBITDA_winsorized_z")
factors_to_keep.remove("RD_P_winsorized_z")

# cleans out any rows missing the dependent or kept factors
train_df = train_df.dropna(subset=['ret_f1'] + factors_to_keep)

# Run monthly cross‐sectionals again, but only on the `factors_to_keep`
monthly_final = []
for yyyymm, grp in train_df.groupby('yyyymm'):
    grp = grp.dropna(subset=['ret_f1'] + factors_to_keep)
    if len(grp) < len(factors_to_keep) + 1:
        continue

    Y = grp['ret_f1']
    X = sm.add_constant(grp[factors_to_keep])   # intercept + kept factors
    model = sm.OLS(Y, X, missing='drop').fit()

    # store the full set of params, plus the month
    row = model.params.to_dict()
    row['yyyymm'] = yyyymm
    monthly_final.append(row)

# Build DataFrame and compute new averages & t‐stats
df_final_coefs = (
    pd.DataFrame(monthly_final)
      .set_index('yyyymm')
      .sort_index()
)

T2 = len(df_final_coefs)
means2  = df_final_coefs.mean()
stds2   = df_final_coefs.std()
se2     = stds2 / np.sqrt(T2)
t2      = means2 / se2

final_summary = pd.DataFrame({
    'avg_premia': means2,
    'std_err':    se2,
    't_stat':     t2,
    'n_months':   T2
}).round(4)

print("\n✔︎ Final Fama–MacBeth (using kept factors):")
print(final_summary.loc[factors_to_keep + ['const']])

In [ ]:
# step 10 - make out os sample dataframe
TRAIN_CUTOFF = 138.0

test_df = (
    filtered_merged_df_factors
    .loc[filtered_merged_df_factors.monthid > TRAIN_CUTOFF]
    .dropna(subset=['ret_f1'] + factors_to_keep)
    .reset_index(drop=True)
)

print(f"Out-of-sample covers months > {TRAIN_CUTOFF}, with {len(test_df)} rows.")

In [ ]:
# This will basically pull in the final in-sample premia & t-stats
#    (computed in Step 9, in final_summary)
premia = final_summary['avg_premia'].to_dict()
tstats = final_summary['t_stat']  .to_dict()

# Predicted‐return approach
# Vectorized formula: α + Σ β_j · F_j
test_df['pred_ret'] = premia['const']
for f in factors_to_keep:
    test_df['pred_ret'] += premia[f] * test_df[f]

# 3B) T-stat‐weighted score approach
# Score = Σ (z_score_f × t_stat_f)
test_df['score'] = 0.0
for f in factors_to_keep:
    test_df['score'] += tstats[f] * test_df[f]

In [ ]:
# Function that ranks them into deciles within each month
def assign_decile(x, col):
    # pd.qcut needs at least as many unique values as bins, should be fine
    return pd.qcut(x[col], 10, labels=False, duplicates='drop') + 1

test_df['decile_pred'] = (
    test_df
    .groupby('yyyymm', group_keys=False)
    .apply(lambda g: assign_decile(g, 'pred_ret'))
) # tdoes it for ret

test_df['decile_score'] = (
    test_df
    .groupby('yyyymm', group_keys=False)
    .apply(lambda g: assign_decile(g, 'score'))
) # does it for the score values

# table outputs
condensed_full_df = test_df[['monthid', 'yyyymm','permno','pred_ret','decile_pred','score','decile_score', 'ret_f1']]

In [ ]:
# Step 12a: Build our decile-based portfolios for each month

def build_decile_portfolio(method, df):
    # Which scoring method are we using
    scoring_string = ""
    if (method == "decile_pred"):
       scoring_string = "pred_ret"
    elif (method == "decile_score"):
        scoring_string = "score"

    # For each month, we group together the stocks included in each decile portfolio
    df["monthid" + method] = df["monthid"].astype(str) + df[method].astype(str)
    model_deciles = df.groupby(["monthid" + method])
    decile_dict = {'monthid':[], method:[], 'freq':[], 'return':[], scoring_string:[]}

    # Now we populate and build a dataframe of our 10 decile portfolios
    for name, group in model_deciles:
        decile_dict['monthid'].append(group['monthid'].iloc[0])
        decile_dict[method].append(group[method].iloc[0])
        decile_dict['freq'].append(group.shape[0])
        decile_dict['return'].append(group['ret_f1'].mean())
        decile_dict[scoring_string].append(group[scoring_string].mean())
    deciles_df = pd.DataFrame(decile_dict)
    deciles_df_sorted = deciles_df.sort_values(["monthid", method])
    return deciles_df_sorted
pred_deciles = build_decile_portfolio('decile_pred', condensed_full_df)
score_deciles = build_decile_portfolio('decile_score', condensed_full_df)
pred_deciles

In [ ]:
# Step 11/12b/13 Compute the two 10% decile extreme portfolios, using equal-weighting
# Simultaneously we build the hedge portfolio, which simply has the return difference of our long and short portfolios
def min_max_portfolios(method, df):
    # Construct, 10% long, 10% short and hedge portfolios
    shortPortfolio = df.loc[df[method]==1]
    longPortfolio = df.loc[df[method]==10]
    hedgePortfolio = pd.merge(shortPortfolio,longPortfolio,how='inner', on=["monthid"])
    hedgePortfolio['hedge_ret'] = hedgePortfolio["return_y"]-hedgePortfolio["return_x"]
    hedgeSorted = hedgePortfolio.sort_values(["monthid"])
    # Merge check - t_tests should be the same for each function call
    print(stats.ttest_ind(longPortfolio['return'], shortPortfolio['return']))
    print(stats.ttest_ind(hedgeSorted['return_y'], hedgeSorted['return_x']))
    print("BREAK")
    return [longPortfolio, shortPortfolio, hedgeSorted]
pred_dfs = min_max_portfolios('decile_pred', pred_deciles)
score_dfs = min_max_portfolios('decile_score', score_deciles)
pred_dfs[2] # Hedge df using Method 1 for scoring


In [ ]:
# combine DataFrame for the two types of methods
df_pred = pred_dfs[2][['monthid', 'hedge_ret']].rename(columns={'hedge_ret':'pred_hedge_ret'})
df_score = score_dfs[2][['monthid', 'hedge_ret']].rename(columns={'hedge_ret':'score_hedge_ret'})
df_combined = pd.merge(df_pred, df_score, on='monthid')
df_combined = df_combined.sort_values('monthid').reset_index(drop=True)

# Compute cumulative returns
df_combined['cumret_pred'] = (1 + df_combined['pred_hedge_ret']).cumprod() - 1
df_combined['cumret_score'] = (1 + df_combined['score_hedge_ret']).cumprod() - 1

# Plotting
plt.figure()
plt.plot(df_combined['monthid'], df_combined['cumret_pred'], label='Predicted-Return Hedge')
plt.plot(df_combined['monthid'], df_combined['cumret_score'], label='Score-Based Hedge')
plt.xlabel('MonthID')
plt.ylabel('Cumulative Return')
plt.title('Cumulative Hedge Portfolio Returns: Predicted Return vs Score')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()